# Phase 1 - Data Acquisition & Setup   
**Dataset:** Bank Customer Churn Modelling   
**Goal:** Download raw data, set up project structure, and load CSV into SQLite   
**Input:** `data/Churn_Modellling.csv` (10,000 rows * 14 columns)   
**Output:** `data/churn.db` - `churn_raw` table   
**Status:** In progress   

In [1]:
# Imports
import pandas as pd
import sqlite3
import os

### Loading the `CSV data` to the notebook
- Display `number of the rows and columns` and all the `column names`

In [2]:
# Load the raw data
df = pd.read_csv('../data/Churn_Modelling.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

Shape: (10000, 14)

Columns:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


### Load the CSV data into a `SQLite` database

In [3]:
# Load the CSV data into a SQLite database
conn = sqlite3.connect('../data/churn.db')

df.to_sql('churn_raw', conn, if_exists='replace', index=False)

print ("Table loaded successfully.")
conn.close()

Table loaded successfully.


### Sanity check:     
- Confirm if the data was loaded successfully to the SQLite database

In [4]:
# Sanity check
conn = sqlite3.connect('../data/churn.db')

check = pd.read_sql("SELECT COUNT(*) AS total_rows FROM churn_raw", conn)
print(check)

conn.close()

   total_rows
0       10000


### Quick peek on few details about the data

In [7]:
conn = sqlite3.connect('../data/churn.db')

summary = pd.read_sql("""
SELECT 
    COUNT(*) AS total_customers,
    COUNT(DISTINCT CustomerId) AS unique_customers,
    ROUND(AVG(Age), 1) As avg_age,
    ROUND(AVG(Balance), 2) AS avg_balance,
    ROUND(AVG(CreditScore), 1) AS avg_credit_score,
    SUM(Exited) AS total_churned,
    ROUND(AVG(CAST(Exited AS FLOAT)) * 100, 2) AS churn_rate_percent
FROM churn_raw
""", conn)

print(summary.T)
conn.close()

                           0
total_customers     10000.00
unique_customers    10000.00
avg_age                38.90
avg_balance         76485.89
avg_credit_score      650.50
total_churned        2037.00
churn_rate_percent     20.37


### Explore the columns

In [8]:
conn = sqlite3.connect('../data/churn.db')

df = pd.read_sql("SELECT * FROM churn_raw", conn)

# Data types and null counts
info = pd.DataFrame({
    'dtype':      df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().sum() / len(df) * 100).round(2),
    'sample':     df.iloc[0]
})

print(info)
conn.close()

                   dtype  null_count  null_pct     sample
RowNumber          int64           0       0.0          1
CustomerId         int64           0       0.0   15634602
Surname              str           0       0.0   Hargrave
CreditScore        int64           0       0.0        619
Geography            str           0       0.0     France
Gender               str           0       0.0     Female
Age                int64           0       0.0         42
Tenure             int64           0       0.0          2
Balance          float64           0       0.0        0.0
NumOfProducts      int64           0       0.0          1
HasCrCard          int64           0       0.0          1
IsActiveMember     int64           0       0.0          1
EstimatedSalary  float64           0       0.0  101348.88
Exited             int64           0       0.0          1
